# Homework 3 — Lead Scoring

Machine Learning Zoomcamp 2026

A reproducible solution covering data preparation, correlations, mutual information, logistic regression, feature elimination, and regularization.


## Setup


In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)


NumPy: 2.3.5
pandas: 2.2.3
scikit-learn: 1.8.0


## Load the dataset


In [2]:
DATA_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_PATH = BASE_DIR / "data" / "course_lead_scoring_2026.csv"

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
TARGET = "converted"

print("Dataset shape:", df.shape)
print("\nPreview:")
print(df.head().to_string(index=False))
print("\nColumn types:")
print(df.dtypes.to_string())


Dataset shape: (5000, 9)

Preview:
   lead_source      industry employment_status      location  annual_income  number_of_courses_viewed  interaction_count  lead_score  converted
organic_search    technology          employed        europe        88160.0                         3                  4        0.64          1
  social_media    technology          employed south_america        72688.0                         3                  7        0.72          1
      referral        retail          employed        europe        44697.0                         3                  4        0.58          1
      paid_ads manufacturing           student           NaN            NaN                         3                  6        0.57          0
      referral manufacturing           student north_america        24062.0                         4                  5        0.62          1

Column types:
lead_source                  object
industry                     object
employment_sta

## Prepare the data


In [3]:
categorical_features = df.drop(columns=TARGET).select_dtypes(include="object").columns.tolist()
numerical_features = df.drop(columns=TARGET).select_dtypes(include="number").columns.tolist()
features = df.columns.drop(TARGET).tolist()

missing_before = df.isna().sum()
print("Missing values before preparation:")
print(missing_before.to_string())

df[categorical_features] = df[categorical_features].fillna("NA")
df[numerical_features] = df[numerical_features].fillna(0.0)

assert df[features].isna().sum().sum() == 0
assert df[TARGET].notna().all()
assert set(df[TARGET].unique()) == {0, 1}

print("\nCategorical features:", categorical_features)
print("Numerical features:", numerical_features)
print("Remaining missing predictor values:", int(df[features].isna().sum().sum()))


Missing values before preparation:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0

Categorical features: ['lead_source', 'industry', 'employment_status', 'location']
Numerical features: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
Remaining missing predictor values: 0


## Question 1 — Mode of industry

Compute value counts after filling missing values. The mode is the most frequent category in the full dataset.


In [4]:
industry_counts = df["industry"].value_counts()
answer_q1 = df["industry"].mode().iloc[0]

print(industry_counts.to_string())
print("\nQ1 answer:", answer_q1)


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240

Q1 answer: technology


## Question 2 — Numerical correlations

Calculate Pearson correlations on the prepared full dataset. Compare only the four feature pairs listed in the question.


In [5]:
correlation_matrix = df[numerical_features].corr()
print("Correlation matrix:")
print(correlation_matrix.to_string())

candidate_pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

correlation_results = pd.DataFrame([
    {"feature_1": first, "feature_2": second,
     "correlation": correlation_matrix.loc[first, second]}
    for first, second in candidate_pairs
])

best_pair = correlation_results.loc[correlation_results["correlation"].idxmax()]
answer_q2 = f"{best_pair['feature_1']} and {best_pair['feature_2']}"

print("\nCandidate pairs:")
print(correlation_results.to_string(index=False))
print("\nQ2 answer:", answer_q2)


Correlation matrix:
                          annual_income  number_of_courses_viewed  interaction_count  lead_score
annual_income                  1.000000                  0.161300           0.122842    0.229496
number_of_courses_viewed       0.161300                  1.000000           0.721609    0.757204
interaction_count              0.122842                  0.721609           1.000000    0.915746
lead_score                     0.229496                  0.757204           0.915746    1.000000

Candidate pairs:
               feature_1         feature_2  correlation
       interaction_count        lead_score     0.915746
number_of_courses_viewed        lead_score     0.757204
number_of_courses_viewed interaction_count     0.721609
           annual_income interaction_count     0.122842

Q2 answer: interaction_count and lead_score


## Split the data

Use the exact two calls from the assignment: first reserve 20% for testing, then reserve 25% of the remaining data for validation.
The resulting split is 60% training, 20% validation, and 20% testing.
Extract `converted` into separate target arrays. The test set remains unused for questions 3–6.


In [6]:
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_full_train = df_full_train.reset_index(drop=True).copy()
df_train = df_train.reset_index(drop=True).copy()
df_val = df_val.reset_index(drop=True).copy()
df_test = df_test.reset_index(drop=True).copy()

y_full_train = df_full_train.pop(TARGET).to_numpy()
y_train = df_train.pop(TARGET).to_numpy()
y_val = df_val.pop(TARGET).to_numpy()
y_test = df_test.pop(TARGET).to_numpy()

print("Training rows:", len(df_train))
print("Validation rows:", len(df_val))
print("Test rows:", len(df_test))
print("Predictors:", features)
assert TARGET not in df_train.columns
assert TARGET not in df_val.columns
assert TARGET not in df_test.columns


Training rows: 3000
Validation rows: 1000
Test rows: 1000
Predictors: ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


## Question 3 — Mutual information

Compute mutual information between each categorical predictor and `converted` using only training rows.
Higher mutual information indicates a stronger statistical association with the target. Scores are rounded with Python's `round(score, 2)`.


In [7]:
mutual_information_results = pd.DataFrame([
    {"feature": feature, "mutual_information": mutual_info_score(df_train[feature], y_train)}
    for feature in categorical_features
])
mutual_information_results["rounded_score"] = mutual_information_results["mutual_information"].apply(
    lambda score: round(score, 2)
)
mutual_information_results = mutual_information_results.sort_values(
    "rounded_score", ascending=False
).reset_index(drop=True)

answer_q3 = mutual_information_results.loc[0, "feature"]
print(mutual_information_results.to_string(index=False))
print("\nQ3 answer:", answer_q3)


          feature  mutual_information  rounded_score
      lead_source            0.034346           0.03
employment_status            0.018966           0.02
         industry            0.001995           0.00
         location            0.001043           0.00

Q3 answer: lead_source


## Model training helper

Fit the encoder on training rows only and transform validation rows with the same encoder.
`DictVectorizer` one-hot encodes string values and keeps numerical values as numerical predictors.
Every model uses `solver="liblinear"`, `max_iter=1000`, and `random_state=42`.


In [8]:
def train_and_evaluate(selected_features, C=1.0):
    train_records = df_train[selected_features].to_dict(orient="records")
    val_records = df_val[selected_features].to_dict(orient="records")

    vectorizer = DictVectorizer(sparse=False)
    X_train = vectorizer.fit_transform(train_records)
    X_val = vectorizer.transform(val_records)

    model = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    accuracy = accuracy_score(y_val, predictions)

    return accuracy, model, vectorizer


## Question 4 — Logistic regression accuracy

Train a logistic regression model with all eight predictors and `C=1.0`.
Accuracy is the fraction of validation rows classified correctly. Report the result rounded to two decimal places.


In [9]:
baseline_accuracy, baseline_model, baseline_vectorizer = train_and_evaluate(features)
answer_q4 = round(baseline_accuracy, 2)

print("Encoded feature count:", len(baseline_vectorizer.get_feature_names_out()))
print("Validation accuracy:", baseline_accuracy)
print("Q4 answer:", answer_q4)


Encoded feature count: 28
Validation accuracy: 0.645
Q4 answer: 0.65


## Question 5 — Feature elimination

Remove one original feature at a time and refit both the encoder and the model.
For categorical features, removing the original feature removes all its one-hot columns.
Calculate `baseline_accuracy - accuracy_without_feature` using unrounded accuracies. A negative difference means that removing the feature improved accuracy.
Select the smallest signed difference among the three answer options only.


In [10]:
elimination_results = []

for excluded_feature in features:
    selected_features = [feature for feature in features if feature != excluded_feature]
    accuracy_without, _, _ = train_and_evaluate(selected_features)

    elimination_results.append({
        "excluded_feature": excluded_feature,
        "accuracy_without": accuracy_without,
        "difference": baseline_accuracy - accuracy_without,
    })

elimination_results = pd.DataFrame(elimination_results)
q5_options = ["lead_source", "number_of_courses_viewed", "interaction_count"]
q5_candidates = elimination_results[
    elimination_results["excluded_feature"].isin(q5_options)
]
answer_q5 = q5_candidates.loc[q5_candidates["difference"].idxmin(), "excluded_feature"]

print("All feature elimination results:")
print(elimination_results.to_string(index=False))
print("\nQuestion 5 candidates:")
print(q5_candidates.to_string(index=False))
print("\nQ5 answer:", answer_q5)


All feature elimination results:
        excluded_feature  accuracy_without  difference
             lead_source             0.642       0.003
                industry             0.645       0.000
       employment_status             0.645       0.000
                location             0.645       0.000
           annual_income             0.724      -0.079
number_of_courses_viewed             0.643       0.002
       interaction_count             0.601       0.044
              lead_score             0.644       0.001

Question 5 candidates:
        excluded_feature  accuracy_without  difference
             lead_source             0.642       0.003
number_of_courses_viewed             0.643       0.002
       interaction_count             0.601       0.044

Q5 answer: number_of_courses_viewed


## Question 6 — Regularization

Train models with all predictors using the four specified values of `C`.
`C` is the inverse regularization strength: smaller values imply stronger regularization.
Compare accuracies rounded with `round(accuracy, 3)`. If multiple values tie, choose the smallest `C`.


In [11]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]
regularization_results = []

for C in C_values:
    accuracy, _, _ = train_and_evaluate(features, C=C)
    regularization_results.append({
        "C": C,
        "accuracy": accuracy,
        "rounded_accuracy": round(accuracy, 3),
    })

regularization_results = pd.DataFrame(regularization_results)
best_rounded_accuracy = regularization_results["rounded_accuracy"].max()
answer_q6 = regularization_results.loc[
    regularization_results["rounded_accuracy"] == best_rounded_accuracy, "C"
].min()

print(regularization_results.to_string(index=False))
print("\nQ6 answer:", answer_q6)


       C  accuracy  rounded_accuracy
0.000001     0.598             0.598
0.000010     0.598             0.598
0.000100     0.613             0.613
0.001000     0.645             0.645

Q6 answer: 0.001


## Final answers

The table below contains the six selections for the homework form.
The calculations use the supplied dataset and the specified preprocessing, random seeds, and model parameters.


In [12]:
answers = pd.DataFrame({
    "question": [1, 2, 3, 4, 5, 6],
    "answer": [answer_q1, answer_q2, answer_q3, answer_q4, answer_q5, answer_q6],
})

print(answers.to_string(index=False))


 question                           answer
        1                       technology
        2 interaction_count and lead_score
        3                      lead_source
        4                             0.65
        5         number_of_courses_viewed
        6                            0.001
